Given two pandas DataFrames, write code to: (1) merge and aggregate revenue; (2) produce a 2x2 pivot; (3) compute per-state counts with value_counts, nunique/size; (4) add a binary flag via np.where. Reuse the merged DataFrame across parts (assume it persists between steps).

In [2]:
import pandas as pd

# Create User DataFrame
user_data = {
    'user_id': [101, 102, 103, 104],
    'is_member': [1, 0, 1, 0],
    'state': ['CA', 'NY', 'CA', 'TX'],
    'age': [29, 41, 35, 50]
}
users = pd.DataFrame(user_data)

# Create Orders DataFrame
order_data = {
    'order_id': [7001, 7002, 7003, 7004, 7005, 7006],
    'user_id': [101, 102, 103, 103, 101, 104],
    'channel': ['SMS', 'Email', 'SMS', 'Email', 'Organic', 'SMS'],
    'amount': [12.00, 5.00, 7.00, 4.00, 3.50, 6.00],
    'status': ['delivered', 'delivered', 'delivered', 'delivered', 'delivered', 'undelivered']
}
orders = pd.DataFrame(order_data)





Tasks

Step 1: Merge orders with users on user_id (left join). Compute two outputs: (a) total delivered revenue by channel; (b) delivered revenue by channel restricted to members (is_member==1). Show groupby(...).sum() results as DataFrames.

Step 2: Create a 2x2 pivot of delivered revenue with index=is_member (0/1) and columns=channel in ['SMS','Email'] only, values=amount, aggfunc='sum', fill missing cells with 0. Use pivot_table with aggfunc='sum'.

Step 3: From the merged DataFrame, compute per-state: total orders (size) and unique purchasers (nunique of user_id). Return the top-2 states by total orders using sort_values.

Step 4: Add column high_value_flag = 1 if (user's lifetime delivered amount >= 15) OR (number of delivered SMS orders per user >= 2), else 0. Use np.where and prior groupby aggregations to avoid SettingWithCopy warnings. Show the final head with relevant columns.

In [29]:
import numpy as np
from IPython.display import display
display(
    orders
    .merge(users, on='user_id', how='left' )
    .query('status=="delivered"')
    .groupby('channel',as_index=False)[['amount']]
    .sum()
)

display(
    orders
    .merge(users, on='user_id', how='left' )
    .query('status=="delivered" and is_member==1')
    .groupby('channel',as_index=False)[['amount']]
    .sum()
)

,channel,amount
0,Email,9.0
1,Organic,3.5
2,SMS,19.0


,channel,amount
0,Email,4.0
1,Organic,3.5
2,SMS,19.0


In [30]:
display(
    orders
    .merge(users, on='user_id', how='left' )
    .query('channel=="SMS" or channel=="Email" ')
    .pivot_table(index='is_member', columns='channel' , values='amount', aggfunc='sum', fill_value=0)
    .reset_index()
    .rename_axis(columns=None)
)

,is_member,Email,SMS
0,0,5.0,6.0
1,1,4.0,19.0


In [ ]:
display(
    orders
    .merge(users, on='user_id', how='left' )
    .groupby('state', as_index=False)
    .agg(total_size=('order_id','count'), nunique=('user_id','nunique'))
    .sort_values(by='total_size', ascending=False)
    .head(2)
)

,state,total_size,nunique
0,CA,4,2
1,NY,1,1


In [31]:
import numpy as np
display(
    orders
    .merge(users, on='user_id', how='left' )
    .query('status=="delivered"')
    .assign(life_amount=lambda df_: df_.groupby('user_id')[['amount']].transform('sum'),
            amount_sms=lambda df_: np.where(df_['channel']=='SMS',df_['amount'], pd.NA),
            sms_deliver=lambda df_: df_.groupby('user_id')[['amount_sms']].transform('count'),
            high_value_flag=lambda df_: np.where((df_['life_amount']>=15) | (df_['sms_deliver']>=2), 1, 0)
            )
    .drop(columns=['life_amount', 'amount_sms','sms_deliver'])
)

,order_id,user_id,channel,amount,status,is_member,state,age,high_value_flag
0,7001,101,SMS,12.0,delivered,1,CA,29,1
1,7002,102,Email,5.0,delivered,0,NY,41,0
2,7003,103,SMS,7.0,delivered,1,CA,35,0
3,7004,103,Email,4.0,delivered,1,CA,35,0
4,7005,101,Organic,3.5,delivered,1,CA,29,1
